[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/1-3_%EA%B2%BD%EB%B3%B4%EA%B8%B0%EC%A4%80%EA%B3%BC_%EC%A0%95%EB%B9%84%EB%B9%84%EC%9A%A9.ipynb)

# 1-3 · 경보 기준을 비용으로 정하기

**⏱ 60분** · 모듈 1: 현장 설비군의 고장 데이터 → 경보 → **비용**

## 🎯 이번 시간의 질문

> 보류 섹터가 **몇 개 이상**일 때 디스크를 교체해야 할까요? 그리고 CBM은 정말 **돈이 되나요?**

지난 시간 결론: 기준을 낮추면 고장을 더 많이 잡지만 멀쩡한 디스크도 많이 교체하고, 기준을 높이면 그 반대입니다.
어느 쪽이 나은지는 **"헛교체 1대"와 "놓친 고장 1건" 중 무엇이 더 비싼가**에 달려 있습니다. 이번 시간에는 두 오류에 비용을 붙여 가장 좋은 기준을 찾고,
**사후정비(고장 나면 교체)**, **예방정비(오래된 것 교체)**와 비교합니다.

**이번 시간에 익힐 패턴:** 변수로 가정 두기 · 계산식 세우기 · P4 선그래프로 비교

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/backblaze_2026Q1.csv.gz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
raw = pd.read_csv('data/backblaze_2026Q1.csv.gz',
                  dtype={'serial_number': 'category', 'model': 'category'},
                  parse_dates=['date'])

def alarm_table(raw, signal):
    """디스크마다 고장 여부, 고장일, 첫 경보일, 경보~고장 여유 일수를 정리한다. (1-2와 같은 함수)"""
    drives = raw.groupby('serial_number', observed=True).agg(failed=('failure', 'max'))
    drives['fail_date'] = raw[raw['failure'] == 1].groupby('serial_number', observed=True)['date'].min()
    usable = signal & (raw['failure'] == 0)
    drives['alarm_date'] = raw[usable].groupby('serial_number', observed=True)['date'].min()
    drives['alarm'] = drives['alarm_date'].notna()
    drives['lead_days'] = (drives['fail_date'] - drives['alarm_date']).dt.days
    return drives

print('읽기 완료:', raw.shape)

## 1. 비용 모델 세우기

| 기호 | 뜻 | 이번 가정 |
|---|---|---|
| `Cp` | **계획 교체** 1대 비용 (디스크값 + 작업) | 1 (기준 단위) |
| `Cf` | **돌발 고장** 1건 비용 (디스크값 + 긴급 복구 + 데이터 재구축 + 서비스 영향) | 20 (계획 교체의 20배라고 가정) |
| `lead` | 경보 후 교체 준비에 필요한 일수 | 7일 |

두 전략의 90일 비용은 이렇게 계산합니다.

- **사후정비(RTF, Run-To-Failure)**: 고장 나면 그때 교체 → `Cf × 고장 수`
- **CBM**: 경보가 난 디스크는 모두 계획 교체. 단, 경보가 고장보다 `lead`일 이상 앞선 경우만 고장을 막는다고 봅니다.
  → `Cp × 경보 대수 + Cf × (막지 못한 고장 수)`

⚠️ `Cp=1`, `Cf=20`은 **가정**입니다. 실제 값은 현장마다 다르므로, 뒤에서 가정을 바꿔 결론이 유지되는지 꼭 확인합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
Cp = 1
Cf = 20
lead = 7

result = alarm_table(raw, raw['smart_197_raw'] > 0)
n_failed = result['failed'].sum()
n_alarms = result['alarm'].sum()
prevented = (result['lead_days'] >= lead).sum()

cost_rtf = Cf * n_failed
cost_cbm = Cp * n_alarms + Cf * (n_failed - prevented)
print('사후정비 비용:', cost_rtf)
print('CBM 비용    :', cost_cbm, f'(교체 {n_alarms}대, 막은 고장 {prevented}건)')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 사후정비 4,000, CBM 3,563.

💬 **결과 해설** — 멀쩡할 수도 있는 디스크 1,683대를 교체하는 비용(1,683)을 치르고, 대신 고장 106건(106 × 20 = 2,120)을 막았습니다. 90일 동안 **약 11% 절감**입니다.
그런데 `> 0`이 가장 좋은 기준일까요? 교체 대수를 줄이면 더 절감될 수도 있습니다.

## 2. 기준을 바꿔 가며 비용 계산하기

기준 0, 1, 2, 4, … 512 각각의 경보 결과를 한 번에 만들어 둡니다. (▶ 20초 정도 걸립니다)

In [ ]:
# ▶ 그대로 실행하세요
thresholds = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
tables = {th: alarm_table(raw, raw['smart_197_raw'] > th) for th in thresholds}

def cost_table(tables, Cp, Cf, lead):
    """기준별 경보 대수, 막은 고장, CBM 비용을 표로 만든다."""
    rows = []
    for th, r in tables.items():
        n_alarms = r['alarm'].sum()
        prevented = (r['lead_days'] >= lead).sum()
        rows.append({'기준': th, '경보 대수': n_alarms, '막은 고장': prevented,
                     'CBM 비용': Cp * n_alarms + Cf * (r['failed'].sum() - prevented)})
    return pd.DataFrame(rows)

costs = cost_table(tables, Cp=1, Cf=20, lead=7)
costs

**P4 · 선그래프** — 기준에 따른 CBM 비용을 그리고, 사후정비 비용을 가로선(`axhline`)으로 겹칩니다.
기준이 0, 1, 2, 4, …처럼 두 배씩 커지므로 `.astype(str)`로 글자로 바꿔 x축에 **같은 간격**으로 놓습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
plt.plot(costs['기준'].astype(str), costs['CBM 비용'], marker='o', label='CBM (smart_197 > 기준)')
plt.axhline(Cf * n_failed, color='red', linestyle='--', label='사후정비')
plt.xlabel('경보 기준 (보류 섹터 수)')
plt.ylabel('90일 비용 (계획 교체 1대 = 1)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 비용 곡선이 **U자 모양**입니다.
- 기준이 너무 낮으면(0~4): 교체 대수가 많아 비쌉니다.
- 기준이 너무 높으면(128 이상): 고장을 많이 놓쳐 비쌉니다.
- 가운데 **32 부근에서 최소(2,981)**, 사후정비(4,000)보다 **약 25% 절감**됩니다.

이것이 CBM에서 경보 기준을 정하는 기본 원리입니다. **"탐지율이 가장 높은 기준"도, "헛경보가 가장 적은 기준"도 아니라, 전체 비용이 가장 작은 기준**을 고릅니다.

---
## 3. 실습 과제 (30분)

### 📝 과제 1 · 고장 비용이 달라지면? (민감도 분석)
`Cf`를 **5, 10, 50**으로 바꿔 각각 가장 좋은 기준과 그때의 비용을 찾고, 사후정비와 비교해 표를 채우세요.

| Cf | 사후정비 비용 | 최적 기준 | 최적 CBM 비용 | CBM이 더 싼가? |
|---|---|---|---|---|
| 5 | | | | |
| 10 | | | | |
| 20 | 4,000 | 32 | 2,981 | 예 |
| 50 | | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: c = cost_table(tables, Cp=1, Cf=5, lead=7) 로 표를 만들고,
# c.sort_values('CBM 비용').head(1) 로 가장 싼 행을 봅니다. 사후정비 비용은 Cf * 200.


✏️ **나의 답:** 고장이 비쌀수록 기준은 어느 방향으로 움직이나요? CBM이 오히려 손해인 경우도 있나요?

### 📝 과제 2 · 준비 시간이 길어지면?
해외 조달 등으로 교체 준비에 **14일** 또는 **30일**이 걸린다면 최적 기준과 비용은 어떻게 변하나요? (`Cf=20`)

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: cost_table(tables, Cp=1, Cf=20, lead=14)


✏️ **나의 답:**

### 📝 과제 3 · 예방정비(나이 기준 교체)와 비교 (도전)
"**1월 1일에 사용 6년 이상 디스크를 모두 교체**"하는 예방정비의 비용을 계산해 CBM과 비교하세요. 교체한 디스크의 고장은 모두 막았다고 봅니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: 디스크별 첫날 사용 연수 → age0 = raw.groupby('serial_number', observed=True)['smart_9_raw'].first() / 24 / 365
# 힌트 2: old = age0 >= 6  →  교체 대수 = old.sum(),  막은 고장 = (old & (result['failed'] == 1)).sum()
# 힌트 3: 비용 = Cp × 교체 대수 + Cf × (200 - 막은 고장)


✏️ **나의 답:** 예방정비·사후정비·CBM(최적) 세 전략의 비용을 비교하고, 차이가 나는 이유를 쓰세요.

### 📝 과제 4 · 정비팀장에게 보내는 3줄 보고
지금까지 결과로 **우리 데이터센터에 추천할 교체 규칙**을 3~4문장으로 쓰세요. ① 추천 규칙 ② 근거 숫자 ③ 가정과 한계를 반드시 포함합니다.

✏️ **나의 답:**

---
## ✅ 모듈 1 정리

| 단계 | 한 일 | 핵심 |
|---|---|---|
| 1-1 고장률 | 디스크·일 분모로 AFR 계산 | 고장은 드물고(1.4%/년) 모델마다 20배 차이 |
| 1-2 경보 | `smart_197 > 기준` 규칙의 탐지율·적중률·여유시간 | 탐지율과 헛경보는 교환 관계 |
| 1-3 비용 | 두 오류에 비용을 붙여 최적 기준 선택 | 최적 기준은 **비용 가정과 준비 시간**에 따라 달라진다 |

모듈 1에서는 "값 > 기준"이라는 **규칙**으로 경보를 만들었습니다. 그런데 센서가 여러 개이고, 무엇이 "정상 값"인지 기준을 모른다면 어떻게 할까요?
→ 모듈 2에서는 **정상 데이터로부터 기준을 배우는 이상탐지**와, **어떤 고장인지 알아내는 진단 모델**을 만듭니다.